# LangGraph basics

A graph connects steps with directed edges. Nodes are functions and state holds their data.

Select the course Python kernel and run cells in order. Sections 1–3 need no model or key. Sections 4–5 use the course Ollama key. Complete each TODO before running its test cells.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

## 1. A sequential graph

Our fictional museum opens at 10:00. We will greet a visitor, add the opening time, and compose an answer.

`START → greet → hours → compose → END`

State is a dictionary. `TypedDict` describes its keys and types; it does not validate values at runtime. We provide the visitor's name; the nodes fill the other fields.

In [ ]:
class VisitorState(TypedDict):
    visitor: str
    greeting: str
    opening_hours: str
    answer: str

A node receives the current state and returns only the fields it updates. Other fields remain available. By default, a new value replaces the previous value for that key.

In [ ]:
def greet(state: VisitorState):
    return {"greeting": f"Hello {state['visitor']}!"}

def hours(state: VisitorState):
    return {"opening_hours": "The museum opens at 10:00."}

def compose(state: VisitorState):
    return {"answer": f"{state['greeting']} {state['opening_hours']}"}

Register each function as a node. Edges set the execution order. `START` marks entry; `END` marks completion. `compile()` creates the runnable graph.

In [ ]:
builder = StateGraph(VisitorState)
builder.add_node("greet", greet)
builder.add_node("hours", hours)
builder.add_node("compose", compose)

builder.add_edge(START, "greet")
builder.add_edge("greet", "hours")
builder.add_edge("hours", "compose")
builder.add_edge("compose", END)

graph = builder.compile()
graph.get_graph().print_ascii()


Like a chain, the graph accepts a dictionary through `invoke()`. Here the result is the final state. Change the visitor's name and run again.

In [ ]:
result = graph.invoke({"visitor": "Lea"})
result

In [ ]:
print(result["answer"])

## 2. A conditional path

Choose one greeting from the language in the input: `en` or `it`. The routing function returns a label; the mapping selects the next node.

In [ ]:
class LanguageState(TypedDict):
    visitor: str
    language: str
    greeting: str

def greet_english(state: LanguageState):
    return {"greeting": f"Hello {state['visitor']}!"}

def greet_italian(state: LanguageState):
    return {"greeting": f"Ciao {state['visitor']}!"}

def select_language(state: LanguageState):
    return state["language"]

In [ ]:
builder = StateGraph(LanguageState)
builder.add_node("english", greet_english)
builder.add_node("italian", greet_italian)

builder.add_conditional_edges(
    START, select_language, {"en": "english", "it": "italian"}
)
builder.add_edge("english", END)
builder.add_edge("italian", END)

language_graph = builder.compile()
language_graph.get_graph().print_ascii()


In [ ]:
result = language_graph.invoke({"visitor": "Lea", "language": "it"})
print(result["greeting"])

Try `en`, then add a French greeting: define its node, register it, add `fr` to the mapping, and connect the node to `END`.

In [ ]:
# Add the French path here, then invoke it with language="fr".

## 3. Parallel paths

Greeting the visitor and getting opening hours are independent. Both nodes can run in the same step; `compose` waits for both.

Each branch writes a different state key. Two parallel writes to the same key would need a rule for combining them, called a reducer. We will use reducers for messages in the next chapter.

Here are the same state and functions together for reference.

In [ ]:
class VisitorState(TypedDict):
    visitor: str
    greeting: str
    opening_hours: str
    answer: str

def greet(state: VisitorState):
    return {"greeting": f"Hello {state['visitor']}!"}

def hours(state: VisitorState):
    return {"opening_hours": "The museum opens at 10:00."}

def compose(state: VisitorState):
    return {"answer": f"{state['greeting']} {state['opening_hours']}"}

In [ ]:
builder = StateGraph(VisitorState)
builder.add_node("greet", greet)
builder.add_node("hours", hours)
builder.add_node("compose", compose)

builder.add_edge(START, "greet")
builder.add_edge(START, "hours")
builder.add_edge(["greet", "hours"], "compose")
builder.add_edge("compose", END)

parallel_graph = builder.compile()
parallel_graph.get_graph().print_ascii()


In [ ]:
result = parallel_graph.invoke({"visitor": "Lea"})
print(result["answer"])

## 4. Together: an LLM inside the graph

In Module 1 you kept a list of messages and sent it to the model on each call. Now we put that model call inside a graph with two nodes:

`START → filter_messages → respond → END`

- `filter_messages` keeps only the most recent messages of the list.
- `respond` sends them to the model and returns the reply.

The list still lives outside the graph, as in Module 1: we pass it on every call and append the reply ourselves.

First load the course key and create the LangChain client.

In [ ]:
from pathlib import Path
import yaml

key_file = Path("../../../llm_provider_api_key.yaml")
api_key = yaml.safe_load(key_file.read_text())["ollama"]["api_key"]


from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": "Bearer " + api_key}},
)

The state has three keys: the full `history`, the filtered `context` sent to the model, and the `reply`.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import (
    ChatPromptTemplate, SystemMessagePromptTemplate,
    HumanMessagePromptTemplate, MessagesPlaceholder,
)

class DialogueState(TypedDict):
    history: list
    context: list
    reply: str

MAX_MESSAGES = 6
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a friendly museum guide. Reply briefly."
    ),
    MessagesPlaceholder("messages"),
])

### Write it together

1. `filter_messages(state)`: return `{"context": ...}` with the last `MAX_MESSAGES` messages of `state["history"]`.
2. `respond(state)`: fill `prompt.format_messages(messages=state["context"])`, pass the messages to `llm.invoke`, and return `{"reply": answer.content}`.
3. Register both nodes, connect `START → filter_messages → respond → END` and compile as `dialogue_graph`.

In [ ]:
# TODO: define filter_messages and respond, then build and compile dialogue_graph.

`chat` keeps the list, as in the Module 1 exercise: it appends your message, invokes the graph with the whole history and appends the reply.

In [ ]:
history = []

def chat(text):
    history.append(HumanMessage(text))
    result = dialogue_graph.invoke({"history": history})
    history.append(AIMessage(result["reply"]))
    print(result["reply"])

chat("My name is Lea. My favourite colour is green.")
chat("What is my name and my favourite colour?")

Set `MAX_MESSAGES = 1`, run the cells again from the state definition and ask the same questions. The model now sees only your last message.

## 5. Exercise: a dialogue manager

Build a graph that decides **how** the virtual human feels before it answers:

`START → detect_intent → choose_emotion → respond → END`

1. `detect_intent` classifies the visitor's message: `GREETING`, `QUESTION`, `COMPLAINT`, `THANKS` or `GOODBYE`.
2. `choose_emotion` chooses the emotion of the reply from the avatar catalog, using the message and its intent.
3. `respond` answers the message, expressing that emotion.

Everything you need is below: the schemas, the structured clients, the state and the prompts. The first two nodes use structured output as in Module 1 Chapter 03.

In [ ]:
from typing import Literal
from pydantic import BaseModel

Intent = Literal["GREETING", "QUESTION", "COMPLAINT", "THANKS", "GOODBYE"]
Emotion = Literal["NEUTRAL", "HAPPY", "ANGRY", "SAD", "FEAR", "DISGUST", "LOVE", "SLEEP"]

class IntentResult(BaseModel):
    intent: Intent

class EmotionResult(BaseModel):
    emotion: Emotion

intent_model = llm.with_structured_output(IntentResult, method="function_calling")
emotion_model = llm.with_structured_output(EmotionResult, method="function_calling")

class ManagerState(TypedDict):
    message: str
    intent: str
    emotion: str
    reply: str

intent_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "Classify the visitor intent: GREETING, QUESTION, COMPLAINT, THANKS or GOODBYE. "
        "Always call IntentResult."
    ),
    HumanMessagePromptTemplate.from_template("{message}"),
])
emotion_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "Choose the museum guide's emotion for intent {intent}. "
        "Use NEUTRAL, HAPPY, ANGRY, SAD, FEAR, DISGUST, LOVE or SLEEP. "
        "Always call EmotionResult."
    ),
    HumanMessagePromptTemplate.from_template("{message}"),
])
reply_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a museum guide. Reply briefly with emotion {emotion}. "
        "If museum facts are missing, say you do not know."
    ),
    HumanMessagePromptTemplate.from_template("{message}"),
])


### Your task

Use the three prompts below. `format_messages(**state)` fills each template from the state dictionary.

1. `detect_intent`: send `intent_prompt.format_messages(**state)` to `intent_model.invoke`; return `{"intent": result.intent}`.
2. `choose_emotion`: send `emotion_prompt.format_messages(**state)` to `emotion_model.invoke`; return `{"emotion": result.emotion}`.
3. `respond`: send `reply_prompt.format_messages(**state)` to `llm.invoke`; return `{"reply": answer.content}`.
4. Build `StateGraph(ManagerState)`, register the nodes, connect them in order from `START` to `END` and compile as `manager`.

In [ ]:
# TODO: define detect_intent, choose_emotion and respond, then build and compile manager.

Test your graph with these messages. Each result shows the intent, the emotion and the reply.

In [ ]:
messages = [
    "Good morning!",
    "What time does the museum close?",
    "I waited forty minutes in the queue, this is unacceptable.",
    "Thank you so much for the tour!",
    "Bye, see you next time.",
]

for message in messages:
    result = manager.invoke({"message": message})
    print(result["intent"], "|", result["emotion"], "|", result["reply"])
    print()

## Next: conversation memory

In section 4 the list of messages lived outside the graph. In the next chapter the messages move into the graph state, a reducer appends them and a checkpointer saves them between calls.

References: [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api), [official quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart).